# 🟢 新手模式 | Beginner Mode

**不用写代码。** 拖动滑杆、点“运行实验”，看曲线和回放怎么变，再回答一个小问题，就完成一次探索。

**三步操作：**
1. 点菜单栏 Run ▶▶ Run All Cells（全部运行一次，等默认结果出来）；
2. 拖动下面的滑杆改一个关键参数；
3. 点 **▶ 运行实验**，看对比图和一句话解读。

**诚实说明：** 你看到的数字全部来自真实引擎（不是演示动画），和完整模式是同一口径。
新手模式只写 `labs/data/lite/` 下的完成记录，**不计入教师成绩册**；想拿实验成绩，请接着做对应的完整模式。

# D1 新手模式：放电越快，能放出的电越少？


## 直觉问题

同一块电池（Chen2020 三元锂电芯），0.5C 慢放能放出约 4.8Ah；如果 2 倍速度放电，还能放出这么多吗？

- **直觉 A**：电量是固定的，放多快都一样。
- **直觉 B**：放得越快，浪费越多，实际能用的电越少。

本实验只让你调一个数：**放电倍率**（0.5C / 1C / 2C）。看电压-容量曲线怎么变。

**你的任务在最后**：0.5C、1C、2C 各跑一次，容量分别是多少？


In [ ]:
import sys
sys.path.insert(0, '/home/jovyan/labs')
sys.dont_write_bytecode = True

# 新手驱动：滑杆 -> 真实引擎 -> 对比图（引擎本身不在本本里，全部复用完整模式的同一批代码）
from common.lite_drivers import (
    plot_curves, plot_bars,
)
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
%matplotlib inline

LAST = {}  # 最近一次运行的结果（供“保存心得”格子使用）
print("✓ 新手驱动就绪：下面拖动滑杆、点运行即可")


In [ ]:
from common.lite_drivers import run_d1_lite, d1_baseline, interpret_d1, save_lite_record

crate = widgets.FloatSlider(
    value=1.0, min=0.5, max=2.0, step=0.5,
    description="放电倍率 (C)：", style={"description_width": "120px"},
    layout=widgets.Layout(width="70%"),
)
run_btn = widgets.Button(description="▶ 运行实验", button_style="success")
out = widgets.Output()

def on_run(b):
    with out:
        out.clear_output(wait=True)
        print(f"正在跑真 PyBaMM 仿真（{crate.value}C 恒流放电）……请稍等")
        base = d1_baseline()
        res = run_d1_lite(crate.value)
        print(f"1C 放出容量：{base['capacity']} Ah；{crate.value}C 放出容量："
              f"{res['capacity']} Ah")
        display(plot_curves(
            [(base["cap"], base["volt"], "1C 基线", "-"),
             (res["cap"], res["volt"], f"{crate.value}C 你的", "-")],
            "已放出容量 (Ah)", "端电压 (V)",
            "放电曲线：基线 vs 你的"))
        print("💡 " + interpret_d1(base["capacity"], res))
        save_lite_record("D1", {"c_rate": crate.value},
                         {"capacity": res["capacity"]})
        LAST.clear()
        LAST.update({"exp": "D1", "params": {"c_rate": crate.value},
                     "metrics": {"capacity": res["capacity"]}})

run_btn.on_click(on_run)
display(crate, run_btn, out)
on_run(None)  # 首次自动跑一次（默认 1C，即基线本身）
print("首次运行已完成，记录摘要：", LAST.get("metrics"))


## 🎯 找区间小任务

把倍率依次调到 0.5C、1C、2C，回答下面两个问题（写在下一个格子的心得里）：

1. 三个倍率放出的容量分别是多少？倍率越高，容量是变大还是变小？
2. 看曲线：倍率越高，电压平台是变高还是变低？末端“电压崩落”出现得更早还是更晚？这说明了什么？


In [ ]:
import ipywidgets as widgets
from IPython.display import display
from common.lite_drivers import save_lite_record

note = widgets.Textarea(
    value="",
    placeholder="写一句话：你找到了什么规律？（例如：阈值越大，刹车越早，但……）",
    description="我的心得：",
    layout=widgets.Layout(width="100%", height="80px"),
)
save_btn = widgets.Button(description="💾 保存心得与完成记录", button_style="info")
save_out = widgets.Output()

def _on_save(b):
    with save_out:
        save_out.clear_output(wait=True)
        if not LAST:
            print("请先运行一次实验（上一个格子的“运行实验”按钮），再保存。")
            return
        path = save_lite_record(LAST["exp"], LAST["params"], LAST["metrics"],
                                note=note.value)
        print(f"✅ 已保存：{path}")
        print("新手模式完成记录不进教师成绩册；完整模式的成绩请去完整模式 notebook 完成。")

save_btn.on_click(_on_save)
display(note, save_btn, save_out)


---

## 下一步 | Next

- 完成上面的“找区间”小任务后，把你的心得保存下来（上一个格子）。
- 想拿实验成绩、学完整算法：去做对应的完整模式（同目录下不带 `_lite` 的 notebook）。
- 新手模式的意义是先建立直觉：参数变了、结果往哪个方向变、变化有多大。

完整模式：`D1_pybamm_battery_pro.ipynb`
